In [1]:
import pandas as pd
from multiset import Multiset

In [2]:
def remove_empty_lines(lines):
    return [line for line in lines if line.strip()]

def identify_resolution(v1, v2, resolution):
    v1_lines = normalize_lines(remove_empty_lines(v1.splitlines()))
    v2_lines = normalize_lines(remove_empty_lines(v2.splitlines()))
    resolution_lines = normalize_lines(remove_empty_lines(resolution.splitlines()))

    if v1_lines == resolution_lines:
        return 'V1'
    elif resolution_lines == v2_lines:
        return 'V2'
    elif resolution_lines == v1_lines + v2_lines:
        return 'ConcatV1V2'
    elif resolution_lines == v2_lines + v1_lines:
        return 'ConcatV2V1'
    else:
        v1_ms = Multiset(v1_lines)
        v2_ms = Multiset(v2_lines)
        resolution_ms = Multiset(resolution_lines)
        if len(resolution_ms - (v1_ms + v2_ms)) > 0:
            return 'New code'
        elif len(resolution_lines) == 0:
            return 'None'
        else:
            return 'Combination'
def normalize_line(line):
    return line.replace(" ", "").replace("\t", "").replace("\n", "")

def normalize_lines(lines):
    normalized_lines = []
    for line in lines:
        normalized_lines.append(normalize_line(line))
    return normalized_lines

In [3]:
df = pd.read_json('../data/dataset2_complete.json')
df = df.rename(columns={'all_raw_a':'v1', 'all_raw_b':'v2', 'all_raw_res': 'resolution', 'all_raw_base':'base'})
df

,base,v1,v2,resolution,merge_id,chunk_number
0,"\t\t\ts3Service.putObject(this.name, binm, new...",\t\t\ttry {\n\t\t\ts3Service.putObject(this.na...,\n\t\t\ttry {\n\t\t\t\ts3Service.putObject(thi...,\t\t\ttry {\n\t\t\ts3Service.putObject(this.na...,272341,1
1,"\t\t\tMap<String, String> mp = this.getUserMet...","\t\t\tMap<String, String> mp = this.getUserMet...",\n\t\t\tObjectMetadata omd = s3Service.getObje...,"\t\t\tMap<String, String> mp = this.getUserMet...",272341,2
2,,\t\t\t\t\n,\n\n,\t\t\t\t\n,272341,3
3,\t\t\telse if (ck.isTruncated()) {\n,\t\t\t}\n\t\t\telse if (ck.isTruncated()) {\n,\n\t\t\t} else if (ck.isTruncated()) {\n,\t\t\t} else if (ck.isTruncated()) {\n,272341,4
4,\t\t\t\tkeys.add(obj.getKey());\n,\t\t\t\t\n\t\t\t\tif (obj.getKey().length() > ...,\n\n\t\t\t\tif (obj.getKey().length() > prefix...,\t\t\t\t\n\t\t\t\tif (obj.getKey().length() > ...,272341,5
...,...,...,...,...,...,...
151421,\n\n,,\n if (changedFilter.label === 'ASSET TYP...,if (changedFilter.label === 'ASSET TYPE'...,49676,2
151422,"it('Swap liquidity', async () => {\n cons...","it('Swap liquidity', async () => {\n cons...",\n,,43954,1
151423,"it('Repay with collateral', async () => {\n ...","it('Repay with collateral', async () => {\n ...",\n,,43954,2
151424,.flashLoan(_mockFlashLoanReceiver.addr...,.flashLoan(_mockFlashLoanReceiver.addr...,\n .flashLoan(_mockFlashLoanReceiver.ad...,.flashLoan(\n _mockFlashLoanR...,43954,3


In [4]:
df['chunk_id'] = df.apply(lambda row: str(row['merge_id']) + '-' +str(row['chunk_number']), axis=1)
df

,base,v1,v2,resolution,merge_id,chunk_number,chunk_id
0,"\t\t\ts3Service.putObject(this.name, binm, new...",\t\t\ttry {\n\t\t\ts3Service.putObject(this.na...,\n\t\t\ttry {\n\t\t\t\ts3Service.putObject(thi...,\t\t\ttry {\n\t\t\ts3Service.putObject(this.na...,272341,1,272341-1
1,"\t\t\tMap<String, String> mp = this.getUserMet...","\t\t\tMap<String, String> mp = this.getUserMet...",\n\t\t\tObjectMetadata omd = s3Service.getObje...,"\t\t\tMap<String, String> mp = this.getUserMet...",272341,2,272341-2
2,,\t\t\t\t\n,\n\n,\t\t\t\t\n,272341,3,272341-3
3,\t\t\telse if (ck.isTruncated()) {\n,\t\t\t}\n\t\t\telse if (ck.isTruncated()) {\n,\n\t\t\t} else if (ck.isTruncated()) {\n,\t\t\t} else if (ck.isTruncated()) {\n,272341,4,272341-4
4,\t\t\t\tkeys.add(obj.getKey());\n,\t\t\t\t\n\t\t\t\tif (obj.getKey().length() > ...,\n\n\t\t\t\tif (obj.getKey().length() > prefix...,\t\t\t\t\n\t\t\t\tif (obj.getKey().length() > ...,272341,5,272341-5
...,...,...,...,...,...,...,...
151421,\n\n,,\n if (changedFilter.label === 'ASSET TYP...,if (changedFilter.label === 'ASSET TYPE'...,49676,2,49676-2
151422,"it('Swap liquidity', async () => {\n cons...","it('Swap liquidity', async () => {\n cons...",\n,,43954,1,43954-1
151423,"it('Repay with collateral', async () => {\n ...","it('Repay with collateral', async () => {\n ...",\n,,43954,2,43954-2
151424,.flashLoan(_mockFlashLoanReceiver.addr...,.flashLoan(_mockFlashLoanReceiver.addr...,\n .flashLoan(_mockFlashLoanReceiver.ad...,.flashLoan(\n _mockFlashLoanR...,43954,3,43954-3


In [5]:
def get_conflict_text(v1, v2):
    return '<<<<<<<\n' + v1 + '=======\n' + v2 + '>>>>>>>\n'
def get_conflict_text_diff3(v1, v2, base):
    return '<<<<<<<\n' + v1 + '|||||||\n' + base + '=======\n' + v2 + '>>>>>>>\n'

In [45]:
chunk_id = '21260-1'

In [46]:
df[df['chunk_id']==chunk_id]

,base,v1,v2,resolution,merge_id,chunk_number,chunk_id
151142,templates.Kits = mockKitHelper;\n ...,// templates.Kits = mockKitHelper;\n ...,\n templates.Kits = mockKitHelper;\n\n ...,\n // templates.Kits = mockKitHelper;\n...,21260,1,21260-1


In [47]:
chunk = df[df['chunk_id']==chunk_id].iloc[0]
v1 = chunk['v1']
v2 = chunk['v2']
base = chunk['base']
# print(get_conflict_text(v1, v2))
print(get_conflict_text_diff3(v1, v2, base))

<<<<<<<
        // templates.Kits = mockKitHelper;
        done();
|||||||
        templates.Kits = mockKitHelper;
        done();

        templates.Kits = mockKitHelper;

        // Delete existing run folder if necessary
        rimraf(runFolder, function (err: Error): void {
            if (err) {
                done(err);
            } else {
                // Create the run folder for our tests
                wrench.mkdirSyncRecursive(runFolder, 511); // 511 decimal is 0777 octal

                // Prepare the test template (archive it)
                wrench.mkdirSyncRecursive(testTemplateArchiveFolder, 511); // 511 decimal is 0777 octal

                var archive: any = archiver("zip");
                var outputStream: NodeJS.WritableStream = fs.createWriteStream(testTemplateArchive);

                archive.on("error", function (err: Error): void {
                    done(err);
                });

                outputStream.on("close", function (): void {
         

In [48]:
resolution = df[df['chunk_id']==chunk_id]['resolution'].values[0]
print(resolution)


        // templates.Kits = mockKitHelper;

        // Delete existing run folder if necessary
        rimraf(runFolder, function (err: Error): void {
            if (err) {
                done(err);
            } else {
                // Create the run folder for our tests
                wrench.mkdirSyncRecursive(runFolder, 511); // 511 decimal is 0777 octal

                // Prepare the test template (archive it)
                wrench.mkdirSyncRecursive(testTemplateArchiveFolder, 511); // 511 decimal is 0777 octal

                var archive: any = archiver("zip");
                var outputStream: NodeJS.WritableStream = fs.createWriteStream(testTemplateArchive);

                archive.on("error", function (err: Error): void {
                    done(err);
                });

                outputStream.on("close", function (): void {
                    done();
                });

                archive.pipe(outputStream);
                archive.directory(testTempla

In [10]:
identify_resolution(v1, v2, resolution)

'Combination'

In [11]:
normalize_line(v1) == normalize_line(resolution)

False

In [12]:
print(len(v1))
print(len(v2))
print(len(resolution))

190
105
150
